[Q 1050. Actors and Directors Who Cooperated At Least Three Times](https://leetcode.com/problems/actors-and-directors-who-cooperated-at-least-three-times/description/)

**Solution:**




In [ ]:
# SQL
SELECT actor_id, director_id
FROM ActorDirector
GROUP BY actor_id, director_id
HAVING COUNT(*) >= 3;

# pandas
import pandas as pd

def actors_and_directors(actor_director: pd.DataFrame) -> pd.DataFrame:
    counts = actor_director.groupby(["actor_id", "director_id"]).size().reset_index(name="count")
    result = counts[counts["count"] >= 3][["actor_id", "director_id"]]
    return result


[Q 1667. Fix Names in a Table](https://leetcode.com/problems/fix-names-in-a-table/description/)

**Solution:**

In [ ]:
# SQL
SELECT user_id,
CONCAT(UPPER(SUBSTRING(name, 1, 1)),
        LOWER(SUBSTRING(name, 2, LENGTH(name)))
        ) AS name
FROM Users

# pandas
import pandas as pd

def fix_names(users: pd.DataFrame) -> pd.DataFrame:
    users["name"] = users["name"].str.capitalize()
    return users.sort_values("user_id")[["user_id", "name"]]

[Q 175. Combine Two Tables](https://leetcode.com/problems/combine-two-tables/description/)

**Solution:**

In [ ]:
# SQL
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person as P LEFT JOIN
Address as a ON
p.personId = a.personId

# pandas
import pandas as pd

def combine_two_tables(person: pd.DataFrame, address: pd.DataFrame) -> pd.DataFrame:
    result = person.merge(address, on="personId", how="left")
    return result[["firstName", "lastName", "city", "state"]]

[Q 176. Second Highest Salary](https://leetcode.com/problems/second-highest-salary/description/)

**Solution:**

In [ ]:
# SQL
SELECT
    (SELECT DISTINCT salary
     FROM Employee
     ORDER BY salary DESC
     LIMIT 1 OFFSET 1) AS SecondHighestSalary;

# pandas
import pandas as pd

def second_highest_salary(employee: pd.DataFrame) -> pd.DataFrame:
    salaries = employee['salary'].drop_duplicates().sort_values(ascending=False)
    second = salaries.iloc[1] if len(salaries) > 1 else None
    return pd.DataFrame({'SecondHighestSalary': [second]})

[Q 1327. List the Products Ordered in a Period](https://leetcode.com/problems/list-the-products-ordered-in-a-period/description/)

**Solution:**

In [ ]:
# SQL

SELECT p.product_name,
       SUM(o.unit) AS unit
FROM Products p
JOIN Orders o
  ON p.product_id = o.product_id
WHERE YEAR(o.order_date) = 2020
  AND MONTH(o.order_date) = 2
GROUP BY p.product_name
HAVING SUM(o.unit) >= 100;

# pandas
import pandas as pd

def list_products(products: pd.DataFrame, orders: pd.DataFrame) -> pd.DataFrame:
    orders["order_date"] = pd.to_datetime(orders["order_date"])

    # filter February 2020
    feb_orders = orders[
        (orders["order_date"].dt.year == 2020) &
        (orders["order_date"].dt.month == 2)
    ]

    # group by product_id and sum units
    totals = feb_orders.groupby("product_id", as_index=False)["unit"].sum()

    # keep those with ≥100 units
    totals = totals[totals["unit"] >= 100]

    # join with products to get names
    result = totals.merge(products, on="product_id")[["product_name", "unit"]]
    return result

Q 1378. Replace Employee ID With The Unique Identifier

**Solution:**


In [ ]:
# SQL
SELECT eu.unique_id,
       e.name
FROM Employees e
LEFT JOIN EmployeeUNI eu
  ON e.id = eu.id;

# pandas
import pandas as pd

def replace_employee_id(employees: pd.DataFrame, employee_uni: pd.DataFrame) -> pd.DataFrame:
    result = employees.merge(employee_uni, on="id", how="left")[["unique_id", "name"]]
    return result

Q 550. Game Play Analysis IV

**Solution:**


In [ ]:
# SQL
SELECT
    ROUND(
        COUNT(DISTINCT a.player_id) /
        (SELECT COUNT(DISTINCT player_id) FROM Activity),
        2
    ) AS fraction
FROM Activity a
JOIN (
    SELECT player_id, MIN(event_date) AS first_login
    FROM Activity
    GROUP BY player_id
) f
ON a.player_id = f.player_id
AND DATEDIFF(a.event_date, f.first_login) = 1;

# pandas
import pandas as pd

def gameplay_analysis(activity: pd.DataFrame) -> pd.DataFrame:
    activity["event_date"] = pd.to_datetime(activity["event_date"])

    # Find first login date per player
    first_login = activity.groupby("player_id")["event_date"].min().reset_index(name="first_login")

    # Merge to compare each player's events with their first login
    merged = activity.merge(first_login, on="player_id")

    # Keep players who logged in exactly one day after first login
    next_day = merged[merged["event_date"] == merged["first_login"] + pd.Timedelta(days=1)]

    # Compute fraction
    next_day_count = next_day["player_id"].nunique()
    total_players = activity["player_id"].nunique()
    fraction = round(next_day_count / total_players, 2)

    return pd.DataFrame({"fraction": [fraction]})

In [ ]:
# SQL
SELECT
    p.project_id,
    ROUND(AVG(e.experience_years), 2) AS average_years
FROM Project p
JOIN Employee e
  ON p.employee_id = e.employee_id
GROUP BY p.project_id;

# pandas
import pandas as pd

def project_employees_i(project: pd.DataFrame, employee: pd.DataFrame) -> pd.DataFrame:
    merged = project.merge(employee, on="employee_id")
    result = (
        merged.groupby("project_id", as_index=False)["experience_years"]
        .mean()
        .round(2)
        .rename(columns={"experience_years": "average_years"})
    )
    return result

Q 185. Department Top Three Salaries

**Solution:**

In [ ]:
# SQL
SELECT
    d.name AS Department,
    e.name AS Employee,
    e.salary AS Salary
FROM (
    SELECT
        e.*,
        DENSE_RANK() OVER (
            PARTITION BY e.departmentId
            ORDER BY e.salary DESC
        ) AS rnk
    FROM Employee e
) e
JOIN Department d
  ON e.departmentId = d.id
WHERE e.rnk <= 3;

# pandas
import pandas as pd

def top_three_salaries(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
    employee["rnk"] = employee.groupby("departmentId")["salary"] \
                              .rank(method="dense", ascending=False)

    # keep only top 3
    top = employee[employee["rnk"] <= 3]

    # join with department to get department names
    result = top.merge(department, left_on="departmentId", right_on="id") \
                [["name_y", "name_x", "salary"]] \
                .rename(columns={"name_y": "Department", "name_x": "Employee", "salary": "Salary"})
    return result